In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from datetime import datetime, timedelta, timezone

import polars as pl

from backtester import io, samplers
from backtester.dtypes import SpotInstrument, OptionInstrument

## Sample data

In [3]:
t0 = datetime(2025, 1, 1, tzinfo=timezone.utc)
tf = datetime(2025, 3, 31, tzinfo=timezone.utc)
dt = timedelta(hours=1)

In [4]:
lf_rate = samplers.get_path_rate(t0, tf, dt)
paths_mark = samplers.get_paths_mark(t0, tf, dt, names=["btc"], s0=[100_000], mu=[0.10], sigma=[0.50])
lf_spot = samplers.to_bars_spot(paths_mark, exchanges="drbt", quotes="usd")
lf_option = samplers.to_bars_option(paths_mark, "drbt", "btc", "usd", rules=[samplers._WEEKLY])

In [5]:
lf_rate.collect()

time_start,time_end,rate
"datetime[μs, UTC]","datetime[μs, UTC]",f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,0.05
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,0.050149
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,0.049953
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,0.050049
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,0.049847
…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,0.047558
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,0.047537
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,0.04748


In [6]:
lf_spot.collect()

time_start,time_end,base,px_mark,exchange,quote,px_bid,px_ask
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,str,f64,f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,"""btc""",100673.710205,"""drbt""","""usd""",99666.973103,101680.447307
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,"""btc""",99951.627551,"""drbt""","""usd""",98952.111276,100951.143827
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,"""btc""",99308.625518,"""drbt""","""usd""",98315.539263,100301.711773
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,"""btc""",99450.223732,"""drbt""","""usd""",98455.721495,100444.725969
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,"""btc""",99580.563351,"""drbt""","""usd""",98584.757718,100576.368985
…,…,…,…,…,…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""btc""",120682.671445,"""drbt""","""usd""",119475.844731,121889.49816
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""btc""",120408.70239,"""drbt""","""usd""",119204.615366,121612.789414
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""btc""",120146.767004,"""drbt""","""usd""",118945.299334,121348.234674


In [7]:
lf_option.collect()

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,iv_bid,iv_ask,iv_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64
2025-01-03 08:00:00 UTC,2025-01-03 09:00:00 UTC,"""drbt""","""btc""","""usd""",120000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 09:00:00 UTC,2025-01-03 10:00:00 UTC,"""drbt""","""btc""","""usd""",120000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 10:00:00 UTC,2025-01-03 11:00:00 UTC,"""drbt""","""btc""","""usd""",120000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 11:00:00 UTC,2025-01-03 12:00:00 UTC,"""drbt""","""btc""","""usd""",120000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 12:00:00 UTC,2025-01-03 13:00:00 UTC,"""drbt""","""btc""","""usd""",120000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
…,…,…,…,…,…,…,…,…,…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""drbt""","""btc""","""usd""",93000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""p""",0.99,1.01,1.0
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""drbt""","""btc""","""usd""",93000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""p""",0.99,1.01,1.0
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""drbt""","""btc""","""usd""",93000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""p""",0.99,1.01,1.0


## `_get_lf_priced`

In [9]:
lf_priced = io._build_lf_priced(
    lf_rate, lf_spot, lf_option,
    option_exchange="drbt",
    option_base="btc",
    option_quote="usd",
    spot_exchange="drbt",
    spot_base="btc",
    spot_quote="usd",
)
lf_priced.collect()

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,spot,rate,iv_bid,iv_ask,iv_mark,px_bid,px_ask,px_mark,delta,gamma,vega,theta,rho
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-03-30 23:00:00 UTC,2025-03-31 00:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",119235.833694,0.047638,0.99,1.01,1.0,423.953358,462.113127,442.827887,0.078722,0.000012,1907.988445,-80784.179954,106.179976
2025-03-30 22:00:00 UTC,2025-03-30 23:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",119431.526577,0.047588,0.99,1.01,1.0,448.300118,487.92553,467.905943,0.082047,0.000011,1981.270593,-83093.979085,111.845899
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",120146.767004,0.04748,0.99,1.01,1.0,518.35075,561.893549,539.913662,0.091756,0.000012,2177.139947,-90461.99797,126.864962
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",120408.70239,0.047537,0.99,1.01,1.0,552.541353,597.941604,575.032272,0.0962,0.000013,2270.012582,-93448.632699,134.462121
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",120682.671445,0.047558,0.99,1.01,1.0,589.416519,636.757502,612.877405,0.100902,0.000013,2367.049143,-96550.467264,142.57291
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-03 12:00:00 UTC,2025-01-03 13:00:00 UTC,"""drbt""","""btc""","""usd""",82000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",98514.934286,0.050516,0.99,1.01,1.0,478.791741,517.899318,498.167259,-0.077768,0.000011,1955.378848,-52133.018406,-151.825797
2025-01-03 11:00:00 UTC,2025-01-03 12:00:00 UTC,"""drbt""","""btc""","""usd""",82000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",99091.609493,0.050679,0.99,1.01,1.0,442.218025,479.531401,460.694753,-0.072276,0.00001,1865.668792,-49442.434203,-142.706783
2025-01-03 10:00:00 UTC,2025-01-03 11:00:00 UTC,"""drbt""","""btc""","""usd""",82000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",99596.766879,0.050787,0.99,1.01,1.0,413.086194,448.911097,430.81758,-0.067831,0.000009,1791.245181,-47186.017495,-135.363006


In [10]:
# Sanity checks: prices positive, Greeks finite
df_priced = lf_priced.collect()
assert (df_priced["px_mark"] >= 0).all(), "px_mark should be non-negative"
assert df_priced["delta"].is_finite().all(), "delta should be finite"
assert df_priced["gamma"].is_finite().all(), "gamma should be finite"
assert df_priced["vega"].is_finite().all(), "vega should be finite"
assert df_priced["theta"].is_finite().all(), "theta should be finite"
assert df_priced["rho"].is_finite().all(), "rho should be finite"
print(f"priced rows: {len(df_priced):,}")

priced rows: 20,800


In [11]:
df_priced.filter(pl.col("px_mark") <= 0).select([
    "time_start", "time_end", "exchange", "base", "quote", "strike", "listing", "expiry", "kind", "px_mark"
])

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,px_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64
2025-03-28 07:00:00 UTC,2025-03-28 08:00:00 UTC,"""drbt""","""btc""","""usd""",120000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""c""",0.0
2025-03-28 07:00:00 UTC,2025-03-28 08:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""p""",0.0
2025-03-28 07:00:00 UTC,2025-03-28 08:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""p""",0.0
2025-03-28 06:00:00 UTC,2025-03-28 07:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""p""",0.0
2025-03-28 05:00:00 UTC,2025-03-28 06:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""p""",0.0
…,…,…,…,…,…,…,…,…,…
2025-01-10 05:00:00 UTC,2025-01-10 06:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
2025-01-10 04:00:00 UTC,2025-01-10 05:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
2025-01-10 03:00:00 UTC,2025-01-10 04:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0


## `get_bars_spot`

In [12]:
spot = SpotInstrument(exchange="drbt", base="btc", quote="usd")

In [13]:
# Full range
io.get_bars_spot(lf_spot, spot).collect()

time_start,time_end,base,px_mark,exchange,quote,px_bid,px_ask
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,str,f64,f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,"""btc""",100673.710205,"""drbt""","""usd""",99666.973103,101680.447307
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,"""btc""",99951.627551,"""drbt""","""usd""",98952.111276,100951.143827
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,"""btc""",99308.625518,"""drbt""","""usd""",98315.539263,100301.711773
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,"""btc""",99450.223732,"""drbt""","""usd""",98455.721495,100444.725969
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,"""btc""",99580.563351,"""drbt""","""usd""",98584.757718,100576.368985
…,…,…,…,…,…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""btc""",120682.671445,"""drbt""","""usd""",119475.844731,121889.49816
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""btc""",120408.70239,"""drbt""","""usd""",119204.615366,121612.789414
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""btc""",120146.767004,"""drbt""","""usd""",118945.299334,121348.234674


In [17]:
# With time filter
start = datetime(2025, 2, 1, tzinfo=timezone.utc)
end = datetime(2025, 2, 28, tzinfo=timezone.utc)
df_spot_filtered = io.get_bars_spot(lf_spot, spot, start_time=start, end_time=end).collect()
assert (df_spot_filtered["time_start"] >= start).all()
assert (df_spot_filtered["time_end"] <= end).all()
df_spot_filtered

time_start,time_end,base,px_mark,exchange,quote,px_bid,px_ask
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,str,f64,f64
2025-02-01 00:00:00 UTC,2025-02-01 01:00:00 UTC,"""btc""",79299.058867,"""drbt""","""usd""",78506.068278,80092.049456
2025-02-01 01:00:00 UTC,2025-02-01 02:00:00 UTC,"""btc""",79651.404851,"""drbt""","""usd""",78854.890802,80447.918899
2025-02-01 02:00:00 UTC,2025-02-01 03:00:00 UTC,"""btc""",80182.839312,"""drbt""","""usd""",79381.010919,80984.667705
2025-02-01 03:00:00 UTC,2025-02-01 04:00:00 UTC,"""btc""",79947.086064,"""drbt""","""usd""",79147.615204,80746.556925
2025-02-01 04:00:00 UTC,2025-02-01 05:00:00 UTC,"""btc""",80124.929497,"""drbt""","""usd""",79323.680202,80926.178792
…,…,…,…,…,…,…,…
2025-02-27 19:00:00 UTC,2025-02-27 20:00:00 UTC,"""btc""",91691.356166,"""drbt""","""usd""",90774.442604,92608.269727
2025-02-27 20:00:00 UTC,2025-02-27 21:00:00 UTC,"""btc""",92372.007934,"""drbt""","""usd""",91448.287854,93295.728013
2025-02-27 21:00:00 UTC,2025-02-27 22:00:00 UTC,"""btc""",92481.032412,"""drbt""","""usd""",91556.222088,93405.842736


## `get_bars_option`

In [18]:
# Pick an arbitrary option from the sampled data
row = lf_option.select(["exchange", "base", "quote", "strike", "listing", "expiry", "kind"]).head(1).collect()
opt = OptionInstrument(
    exchange=row["exchange"].item(),
    base=row["base"].item(),
    quote=row["quote"].item(),
    strike=row["strike"].item(),
    listing=row["listing"].item(),
    expiry=row["expiry"].item(),
    kind=row["kind"].item(),
)
opt

OptionInstrument(exchange='drbt', base='btc', quote='usd', strike=140000.0, listing=datetime.datetime(2025, 3, 28, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), expiry=datetime.datetime(2025, 4, 4, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), kind='c')

In [19]:
# Full range
io.get_bars_option(lf_option, opt).collect()

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,iv_bid,iv_ask,iv_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64
2025-03-30 23:00:00 UTC,2025-03-31 00:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 22:00:00 UTC,2025-03-30 23:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
…,…,…,…,…,…,…,…,…,…,…,…
2025-03-28 12:00:00 UTC,2025-03-28 13:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-28 11:00:00 UTC,2025-03-28 12:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-28 10:00:00 UTC,2025-03-28 11:00:00 UTC,"""drbt""","""btc""","""usd""",140000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0


In [20]:
# With time filter
df_opt_filtered = io.get_bars_option(lf_option, opt, start_time=start, end_time=end).collect()
if len(df_opt_filtered) > 0:
    assert (df_opt_filtered["time_start"] > start).all()
    assert (df_opt_filtered["time_end"] < end).all()
df_opt_filtered

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,iv_bid,iv_ask,iv_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64


## `get_target_option`

In [21]:
target = io.get_target_option(
    lf_rate, lf_spot, lf_option,
    option_exchange="drbt",
    option_base="btc",
    option_quote="usd",
    option_kind="c",
    spot_instrument=spot,
    target_time=datetime(2025, 2, 1, tzinfo=timezone.utc),
    target_delta=0.5,
    target_tenor=timedelta(days=30),
)
target

OptionInstrument(exchange='drbt', base='btc', quote='usd', strike=80000.0, listing=datetime.datetime(2025, 1, 31, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), expiry=datetime.datetime(2025, 2, 7, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), kind='c')

In [22]:
assert isinstance(target, OptionInstrument)
assert target.kind == "c"
assert target.exchange == "drbt"
assert target.base == "btc"
print(f"strike={target.strike}, listing={target.listing}, expiry={target.expiry}")

strike=80000.0, listing=2025-01-31 08:00:00+00:00, expiry=2025-02-07 08:00:00+00:00
